# Part 4 (2/2) — Category Analysis: Map-Reduce & Brand Comparison — **run locally**

This notebook makes **every call to the Metis API** for the category-level
analysis. It does no dataset downloading and no heavy pandas work — it only
reads `category_prep_data.json` (produced by
`category_analysis_prep_kaggle.ipynb`) — which now contains a `"categories"`
array, one entry per eligible category — and loops over **every category**,
turning each one's prepared text batches into LLM calls, exactly like
`run_validation.ipynb` does for Part 1.

**Before running:** put `category_prep_data.json` (downloaded from Kaggle)
in the same folder as this notebook.

**Output:** `category_insights.json` — now also a `"categories"` array (one
report per category, same per-category schema as before), graded next by
`category_judge.ipynb` (which is also local-only and needs no further
splitting).


## بخش ۱ — نصب و ایمپورت

In [1]:
import os
import json
import time
import re

import requests
import pandas as pd


## بخش ۲ — تنظیمات، قیمت‌گذاری و بودجه

In [ ]:
INPUT_PREP_FILE = "category_prep_data.json"   # خروجی category_analysis_prep_kaggle.ipynb
OUTPUT_REPORT_JSON = "category_insights.json"

RUN_MODE = "single"        # "single": فقط یک دسته پردازش شود (سریع، برای تست) | "all": همه‌ی دسته‌ها (کند)
SELECTED_CATEGORY = None   # نام دقیق یا شماره‌ی دسته در لیستی که چاپ می‌شود؛ اگر None بماند و RUN_MODE="single"، همان لحظه از شما پرسیده می‌شود

API_URL = "https://api.metisai.ir/openai/v1/chat/completions"
API_KEY = os.environ.get("METIS_API_KEY", "****")
MODEL_NAME = "gpt-4o-mini"

# ------------------------------------------------------------
# قیمت‌گذاری و بودجه (Placeholder — نرخ عمومی OpenAI؛ قبل از اجرای واقعی
# با نرخ دقیق حساب Metis AI خودتان جایگزین کنید)
# ------------------------------------------------------------
PRICING_PER_1M_TOKENS = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4o": {"input": 2.50, "output": 10.00},
}
MAX_BUDGET_USD = 5.0

total_prompt_tokens = 0
total_completion_tokens = 0
total_cost_usd = 0.0


def estimate_cost(model, prompt_tokens, completion_tokens):
    rates = PRICING_PER_1M_TOKENS.get(model)
    if rates is None:
        return 0.0
    return (prompt_tokens / 1_000_000) * rates["input"] + (completion_tokens / 1_000_000) * rates["output"]


def budget_remaining():
    return total_cost_usd < MAX_BUDGET_USD


## بخش ۳ — بارگذاری داده‌ی آماده‌شده از Kaggle (همه‌ی دسته‌ها)

`category_prep_data.json` اکنون یک لیست از دسته‌ها را زیر کلید
`"categories"` نگه می‌دارد — هر عضو، همان اسکیمایی را دارد که پیش‌تر
مستقیماً در ریشه‌ی فایل بود (`category`, `products_analyzed`,
`negative_comment_batches`, `brand_samples`, `top_brands_stats`, ...).

این سلول همچنان **همه‌ی** دسته‌های موجود در فایل را بارگذاری می‌کند (برای نمایش لیست در بخش بعد)؛ فیلتر کردن به یک دسته یا نگه‌داشتن همه، بر اساس `RUN_MODE` در سلول بعدی انجام می‌شود.


In [3]:
with open(INPUT_PREP_FILE, "r", encoding="utf-8") as f:
    prep_output = json.load(f)

all_categories_prep = prep_output["categories"]

print("Categories loaded from Kaggle prep file:", len(all_categories_prep))
for cat in all_categories_prep:
    print(f"  - {cat['category']}: products={cat['products_analyzed']}, negative_batches={len(cat['negative_comment_batches'])}")


Categories loaded from Kaggle prep file: 182
  - مراقبت پوست: products=2039, negative_batches=20
  - اسباب بازی: products=883, negative_batches=20
  - شامپو و مراقبت مو: products=1721, negative_batches=20
  - اکسسوری زنانه و مردانه: products=971, negative_batches=20
  - لباس مردانه: products=660, negative_batches=20
  - بهداشت و مراقبت بدن: products=1554, negative_batches=20
  - لباس زنانه: products=529, negative_batches=20
  - نوشت افزار: products=878, negative_batches=20
  - برس‌ها و تجهیزات آرایشی: products=881, negative_batches=20
  - بهداشت دهان و دندان: products=1398, negative_batches=20
  - لوازم اصلاح مو: products=1296, negative_batches=20
  - اکسسوری مردانه: products=834, negative_batches=20
  - کتاب شعر و ادبیات: products=523, negative_batches=20
  - دفتر و کاغذ و مقوا: products=704, negative_batches=20
  - بهداشت و زیبایی ناخن: products=726, negative_batches=20
  - کفش مردانه: products=770, negative_batches=20
  - آرایش مو: products=565, negative_batches=20
  - لوازم اداری: 

## بخش ۳-الف — انتخاب دسته برای پردازش (`RUN_MODE`)

اگر `RUN_MODE = "single"` باشد، لیست دسته‌ها چاپ می‌شود و باید یکی را با
نام دقیق یا شماره‌اش انتخاب کنید (یا از قبل در `SELECTED_CATEGORY` ست
کرده باشید تا سؤال پرسیده نشود). اگر `RUN_MODE = "all"` باشد، رفتار قبلی
(پردازش همه‌ی دسته‌ها) اجرا می‌شود.


In [4]:
if RUN_MODE == "all":
    categories_to_process = all_categories_prep
    print(f"RUN_MODE='all' -> همه‌ی {len(categories_to_process)} دسته پردازش می‌شوند (ممکن است طول بکشد).")

elif RUN_MODE == "single":
    print("دسته‌های موجود:")
    for i, cat in enumerate(all_categories_prep):
        print(f"  [{i}] {cat['category']}  (محصولات: {cat['products_analyzed']}, بچ‌های نظر منفی: {len(cat['negative_comment_batches'])})")

    def resolve_category(query, categories):
        query = str(query).strip()
        if query.isdigit():
            idx = int(query)
            if 0 <= idx < len(categories):
                return categories[idx]
            raise ValueError(f"ایندکس نامعتبر: {idx} (باید بین ۰ تا {len(categories) - 1} باشد)")
        exact = [c for c in categories if c["category"] == query]
        if exact:
            return exact[0]
        partial = [c for c in categories if query in c["category"]]
        if len(partial) == 1:
            return partial[0]
        if len(partial) > 1:
            names = "، ".join(c["category"] for c in partial)
            raise ValueError(f"چند دسته با «{query}» مطابقت دارند: {names} — دقیق‌تر بنویسید.")
        raise ValueError(f"دسته‌ای با نام/شماره‌ی «{query}» پیدا نشد.")

    query = "کنسول خانگی"
    if query is None:
        query = input("\nنام دقیق یا شماره‌ی دسته‌ی موردنظر را وارد کنید: ")

    selected_category_prep = resolve_category(query, all_categories_prep)
    categories_to_process = [selected_category_prep]
    print(f"\nانتخاب شد: {selected_category_prep['category']}")

else:
    raise ValueError(f"RUN_MODE نامعتبر: {RUN_MODE!r} (باید 'single' یا 'all' باشد)")


دسته‌های موجود:
  [0] مراقبت پوست  (محصولات: 2039, بچ‌های نظر منفی: 20)
  [1] اسباب بازی  (محصولات: 883, بچ‌های نظر منفی: 20)
  [2] شامپو و مراقبت مو  (محصولات: 1721, بچ‌های نظر منفی: 20)
  [3] اکسسوری زنانه و مردانه  (محصولات: 971, بچ‌های نظر منفی: 20)
  [4] لباس مردانه  (محصولات: 660, بچ‌های نظر منفی: 20)
  [5] بهداشت و مراقبت بدن  (محصولات: 1554, بچ‌های نظر منفی: 20)
  [6] لباس زنانه  (محصولات: 529, بچ‌های نظر منفی: 20)
  [7] نوشت افزار  (محصولات: 878, بچ‌های نظر منفی: 20)
  [8] برس‌ها و تجهیزات آرایشی  (محصولات: 881, بچ‌های نظر منفی: 20)
  [9] بهداشت دهان و دندان  (محصولات: 1398, بچ‌های نظر منفی: 20)
  [10] لوازم اصلاح مو  (محصولات: 1296, بچ‌های نظر منفی: 20)
  [11] اکسسوری مردانه  (محصولات: 834, بچ‌های نظر منفی: 20)
  [12] کتاب شعر و ادبیات  (محصولات: 523, بچ‌های نظر منفی: 20)
  [13] دفتر و کاغذ و مقوا  (محصولات: 704, بچ‌های نظر منفی: 20)
  [14] بهداشت و زیبایی ناخن  (محصولات: 726, بچ‌های نظر منفی: 20)
  [15] کفش مردانه  (محصولات: 770, بچ‌های نظر منفی: 20)
  [16] آرایش مو  (محصولا

## بخش ۴ — تابع فراخوانی API (با ردیابی هزینه) و استخراج امن JSON

In [5]:
def call_llm(prompt, model=MODEL_NAME):
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"

    payload = {
        "model": model,
        "messages": [
            {
                "role": "system",
                "content": "You are a precise data analyst. Always respond with valid JSON only, no extra text.",
            },
            {"role": "user", "content": prompt},
        ],
        "temperature": 0,
    }

    start_time = time.perf_counter()
    response = requests.post(API_URL, headers=headers, json=payload, timeout=300)
    latency = time.perf_counter() - start_time

    response.raise_for_status()
    data = response.json()
    content = data["choices"][0]["message"]["content"]

    usage = data.get("usage", {})
    return content, latency, usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0)


def extract_json(raw_text):
    text = raw_text.strip()
    text = re.sub(r"^```(json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()

    match = re.search(r"(\{.*\}|\[.*\])", text, flags=re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in response: {raw_text[:200]}")
    return json.loads(match.group(0))


## بخش ۵ — مرحله‌ی Map: استخراج موضوعات شکایت از هر Batch

In [6]:
def build_map_prompt(batch, category_name):
    comments_block = []
    for record in batch:
        comments_block.append(
            f"[ID {record['id']}] Title: {record['title']} | Body: {record['body']} | "
            f"Disadvantages: {record['disadvantages']} | Recommendation: {record['recommendation_status']}"
        )
    comments_text = "\n".join(comments_block)

    prompt = f"""
شما یک تحلیل‌گر داده هستید که نظرات منفی مشتریان درباره‌ی محصولات دسته‌ی
"{category_name}" را بررسی می‌کند.

از بین نظرات زیر، موضوعات تکرارشونده‌ی شکایت یا نارضایتی (از جمله ایراد
در ویژگی‌های خاص محصول) را استخراج کن.

فقط بر اساس همین نظرات عمل کن؛ موضوعی را که در نظرات نیست حدس نزن.

فقط یک آرایه‌ی JSON برگردان، دقیقاً با این ساختار، بدون هیچ توضیح اضافه:

[
  {{
    "theme": "نام کوتاه و مشخص موضوع شکایت",
    "count_in_batch": <عدد>,
    "example_comment_ids": [<چند شناسه‌ی نمونه از همین Batch>]
  }}
]

نظرات:
{comments_text}
"""
    return prompt


## بخش ۶ — مرحله‌ی Reduce: تابع تجمیع موضوعات شکایت

خروجی همه‌ی Batchهای یک دسته یک‌بار دیگر به مدل داده می‌شود تا موضوعات
مشابه ادغام و Top-8 موضوع نهایی استخراج شود. سپس متن چند نظر نمونه برای هر
موضوع پیوست می‌شود. این بخش فقط توابع را تعریف می‌کند؛ فراخوانی واقعی در
تابع پردازش هر دسته (بخش ۸) انجام می‌شود.


In [7]:
def build_reduce_prompt(batch_theme_results, category_name):
    combined = []
    for batch_themes in batch_theme_results:
        combined.extend(batch_themes)
    combined_text = json.dumps(combined, ensure_ascii=False, indent=2)

    prompt = f"""
شما نتایج مرحله‌ی Map را برای دسته‌ی محصول "{category_name}" در اختیار دارید:
فهرستی از موضوعات شکایت که از چند Batch نظر جداگانه استخراج شده‌اند.

این فهرست‌ها را ترکیب کن: موضوعات مشابه یا تکراری را یکی کن، تعداد
تکرارها (count_in_batch) را برای موضوعات یکسان جمع بزن، و در نهایت
Top 8 موضوع پرتکرارترین شکایت را برگردان.

فقط بر اساس داده‌های زیر عمل کن؛ موضوع جدیدی که در داده‌ها نیست اضافه نکن.

فقط یک آرایه‌ی JSON برگردان، دقیقاً با این ساختار:

[
  {{
    "theme": "نام نهایی موضوع",
    "total_mentions": <مجموع تکرار>,
    "representative_comment_ids": [<چند شناسه‌ی نمونه>]
  }}
]

داده‌های Map:
{combined_text}
"""
    return prompt


def attach_snippets(theme_list, lookup, max_snippets=3):
    for theme in theme_list:
        snippets = []
        for comment_id in theme.get("representative_comment_ids", [])[:max_snippets]:
            info = lookup.get(comment_id)
            if info:
                snippet = f"{info['title']} — {info['body']}"
                snippets.append({"comment_id": comment_id, "snippet": snippet[:300]})
        theme["representative_comments"] = snippets
    return theme_list


## بخش ۷ — تابع ساخت Prompt مقایسه‌ی برند

دامنه محدود است (چند برند اصلی، آماده‌شده روی Kaggle برای هر دسته)، پس
یک فراخوانی واحد به‌ازای هر دسته کافی است — مدل فقط بر اساس آمار کمّی
واقعی و نمونه نظر واقعی همان دسته پاسخ می‌دهد.


In [8]:
def build_brand_comparison_prompt(top_brands_stats, brand_samples, category_name):
    stats_text = json.dumps(top_brands_stats, ensure_ascii=False)
    samples_text = json.dumps(brand_samples, ensure_ascii=False, indent=2)

    prompt = f"""
شما در حال مقایسه‌ی برندهای اصلی دسته‌ی "{category_name}" هستید.

آمار کمّی هر برند:
{stats_text}

نمونه نظرات واقعی مثبت و منفی هر برند:
{samples_text}

بر اساس فقط همین اطلاعات (آمار + نمونه نظرات)، برای هر برند یک خلاصه‌ی
کوتاه (۲ تا ۳ جمله) از نقاط قوت و ضعف آن از دید مشتریان بنویس. اگر
شواهد کافی برای یک برند نیست، همین را صریحاً بگو.

هیچ ادعایی که در آمار یا نمونه نظرات نیست اضافه نکن.

فقط یک شیء JSON با این دقیق ساختار برگردان (کلید = نام برند):

{{
  "نام برند": "خلاصه ۲ تا ۳ جمله‌ای"
}}
"""
    return prompt


## بخش ۸ — تابع پردازش کامل یک دسته (Map → Reduce → مقایسه‌ی برند)

مراحل Map، Reduce و مقایسه‌ی برند که پیش‌تر یک‌بار و برای یک دسته اجرا
می‌شدند، در تابع `run_category_analysis` جمع شده‌اند تا در حلقه‌ی بخش
بعد، بدون تغییر، روی تک‌تک دسته‌ها اجرا شوند. بودجه (`MAX_BUDGET_USD`)
میان همه‌ی دسته‌ها **مشترک** است؛ با تمام‌شدن بودجه، همان‌جا (چه وسط
Batchهای یک دسته، چه قبل از شروع دسته‌ی بعدی) متوقف می‌شود.


In [9]:
def run_category_analysis(category_prep):
    global total_prompt_tokens, total_completion_tokens, total_cost_usd

    category_name = category_prep["category"]
    negative_comment_batches = category_prep["negative_comment_batches"]
    brand_samples = category_prep["brand_samples"]
    top_brands_stats = category_prep["top_brands_stats"]

    comment_lookup = {
        record["id"]: {"title": record["title"], "body": record["body"]}
        for batch in negative_comment_batches
        for record in batch
    }

    # --- Map ---
    batch_theme_results = []
    for index, batch in enumerate(negative_comment_batches, start=1):
        if not budget_remaining():
            print(f"    Budget limit reached — stopping map step for '{category_name}' "
                  f"before batch {index}/{len(negative_comment_batches)}.")
            break
        prompt = build_map_prompt(batch, category_name)
        try:
            raw_response, latency, prompt_tokens, completion_tokens = call_llm(prompt)
            call_cost = estimate_cost(MODEL_NAME, prompt_tokens, completion_tokens)
            total_prompt_tokens += prompt_tokens
            total_completion_tokens += completion_tokens
            total_cost_usd += call_cost
            batch_theme_results.append(extract_json(raw_response))
        except Exception as e:
            print(f"    ERROR (map batch {index}): {e}")

    # --- Reduce ---
    top_complaint_themes = []
    if batch_theme_results and budget_remaining():
        try:
            reduce_prompt = build_reduce_prompt(batch_theme_results, category_name)
            raw_response, latency, prompt_tokens, completion_tokens = call_llm(reduce_prompt)
            call_cost = estimate_cost(MODEL_NAME, prompt_tokens, completion_tokens)
            total_prompt_tokens += prompt_tokens
            total_completion_tokens += completion_tokens
            total_cost_usd += call_cost
            top_complaint_themes = extract_json(raw_response)
            top_complaint_themes = attach_snippets(top_complaint_themes, comment_lookup)
        except Exception as e:
            print(f"    ERROR (reduce): {e}")

    # --- Brand comparison ---
    brand_comparison_narrative = {}
    if brand_samples and budget_remaining():
        try:
            prompt = build_brand_comparison_prompt(top_brands_stats, brand_samples, category_name)
            raw_response, latency, prompt_tokens, completion_tokens = call_llm(prompt)
            call_cost = estimate_cost(MODEL_NAME, prompt_tokens, completion_tokens)
            total_prompt_tokens += prompt_tokens
            total_completion_tokens += completion_tokens
            total_cost_usd += call_cost
            brand_comparison_narrative = extract_json(raw_response)
        except Exception as e:
            print(f"    ERROR (brand comparison): {e}")

    return {
        "category": category_name,
        "products_analyzed": category_prep["products_analyzed"],
        "comments_analyzed_for_themes": category_prep["comments_analyzed_for_themes"],
        "high_volume_low_recommendation_products": category_prep["high_volume_low_recommendation_products"],
        "top_complaint_themes": top_complaint_themes,
        "brand_comparison": {
            "stats": top_brands_stats,
            "sample_comments": brand_samples,
            "narrative": brand_comparison_narrative,
        },
    }


## بخش ۹ — اجرای تحلیل برای دسته(های) انتخاب‌شده (با Budget Guard)

حلقه‌ی زیر روی `categories_to_process` اجرا می‌شود — که بسته به `RUN_MODE`
یا شامل یک دسته‌ی انتخاب‌شده است یا همه‌ی دسته‌ها. قبل از شروع هر دسته،
بودجه‌ی تجمعی چک می‌شود؛ با رسیدن به سقف، بقیه‌ی دسته‌ها بدون فراخوانی رد
می‌شوند.


In [10]:
all_category_reports = []

for idx, category_prep in enumerate(categories_to_process, start=1):
    category_name = category_prep["category"]

    if not budget_remaining():
        print(f"\nBudget limit reached (${total_cost_usd:.4f} >= ${MAX_BUDGET_USD:.2f}). "
              f"Skipping remaining categories ({len(categories_to_process) - idx + 1} left), "
              f"starting with '{category_name}'.")
        break

    print(f"\n[{idx}/{len(categories_to_process)}] Processing category: {category_name}")
    report = run_category_analysis(category_prep)
    all_category_reports.append(report)
    print(f"  -> themes: {len(report['top_complaint_themes'])} | "
          f"brands: {len(report['brand_comparison']['narrative'])} | "
          f"running cost: ${total_cost_usd:.4f}")

print(f"\nCategories fully processed: {len(all_category_reports)} از {len(categories_to_process)}")



[1/1] Processing category: کنسول خانگی
  -> themes: 7 | brands: 3 | running cost: $0.0149

Categories fully processed: 1 از 1


## بخش ۱۰ — خلاصه‌ی مصرف توکن و هزینه


In [11]:
print("=" * 40)
print(f"Total prompt tokens:     {total_prompt_tokens}")
print(f"Total completion tokens: {total_completion_tokens}")
print(f"Total tokens:            {total_prompt_tokens + total_completion_tokens}")
print(f"Estimated total cost:    ${total_cost_usd:.4f}")
print(f"Budget limit:            ${MAX_BUDGET_USD:.2f}")
print(f"Remaining budget:        ${MAX_BUDGET_USD - total_cost_usd:.4f}")
print("=" * 40)


Total prompt tokens:     73674
Total completion tokens: 6489
Total tokens:            80163
Estimated total cost:    $0.0149
Budget limit:            $5.00
Remaining budget:        $4.9851


## بخش ۱۱ — ساخت و ذخیره‌ی گزارش نهایی (`category_insights.json`، همه‌ی دسته‌ها)

مانند فایل ورودی، خروجی هم اکنون یک آرایه‌ی `"categories"` است؛ اسکیمای
هر آیتم دقیقاً همانی است که پیش‌تر برای یک دسته تولید می‌شد. نوت‌بوک بعدی
(`category_judge.ipynb`) باید روی `data["categories"]` حلقه بزند.


In [12]:
output_data = {
    "categories": all_category_reports,
}

with open(OUTPUT_REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"Saved: {OUTPUT_REPORT_JSON}")
print(f"Categories included: {len(all_category_reports)}")


Saved: category_insights.json
Categories included: 1


## بخش ۱۲ — خلاصه‌ی خوانا برای همه‌ی دسته‌ها + گزارش کامل یک دسته‌ی نمونه

چاپ گزارش کامل برای همه‌ی دسته‌ها عملاً غیرقابل‌خواندن می‌شود، پس یک جدول
خلاصه برای همه‌ی دسته‌ها نمایش داده می‌شود و گزارش کامل فقط برای اولین
دسته‌ی پردازش‌شده (به‌عنوان نمونه) چاپ می‌شود.


In [13]:
if all_category_reports:
    summary_rows = []
    for report in all_category_reports:
        summary_rows.append({
            "category": report["category"],
            "products_analyzed": report["products_analyzed"],
            "complaint_themes": len(report["top_complaint_themes"]),
            "high_volume_low_rec_products": len(report["high_volume_low_recommendation_products"]),
            "brands_compared": len(report["brand_comparison"]["narrative"]),
        })
    display(pd.DataFrame(summary_rows))

    sample_report = all_category_reports[0]
    print("=" * 60)
    print(f"SAMPLE REPORT — {sample_report['category']}")
    print("=" * 60)

    print(f"\nProducts analyzed: {sample_report['products_analyzed']}")
    print(f"Comments analyzed for complaint themes: {sample_report['comments_analyzed_for_themes']}")

    print("\n--- Top Complaint Themes ---")
    for theme in sample_report["top_complaint_themes"]:
        print(f"- {theme.get('theme')}  (mentions: {theme.get('total_mentions')})")

    print("\n--- High-Volume / Low-Recommendation Products ---")
    for p in sample_report["high_volume_low_recommendation_products"][:10]:
        print(f"- {p['title_fa']} ({p['Brand']}) | comments={p['comment_count']} | rec_rate={p['recommendation_rate']:.2f}")

    print("\n--- Brand Comparison ---")
    for brand, summary in sample_report["brand_comparison"]["narrative"].items():
        print(f"\n{brand}:\n{summary}")
else:
    print("هیچ دسته‌ای پردازش نشد.")


,category,products_analyzed,complaint_themes,high_volume_low_rec_products,brands_compared
0,کنسول خانگی,40,7,4,3


SAMPLE REPORT — کنسول خانگی

Products analyzed: 40
Comments analyzed for complaint themes: 612

--- Top Complaint Themes ---
- کیفیت پایین  (mentions: 26)
- قیمت بالا  (mentions: 38)
- حافظه کم  (mentions: 29)
- مشکلات عملکرد  (mentions: 8)
- دستگاه معیوب یا دست دوم  (mentions: 10)
- بسته بندی نامناسب  (mentions: 20)
- مشکلات فنی  (mentions: 10)

--- High-Volume / Low-Recommendation Products ---
- کنسول بازی سونی مدل Playstation 4 Slim کد Region 2 CUH-2216B ظرفیت یک ترابایت  (سونی) | comments=200 | rec_rate=0.73
- مجموعه کنسول بازی سونی مدل PlayStation 5 Drive ظرفیت 825 گیگابایت به همراه هدست و پایه شارژر (سونی) | comments=200 | rec_rate=0.83
- کنسول بازی سونی مدل Playstation 4 Slim ریجن 3 کد CUH-2218B ظرفیت 1 ترابایت (سونی) | comments=171 | rec_rate=0.84
- کنسول بازی مدل 4K Game Stick Lite (متفرقه) | comments=101 | rec_rate=0.51

--- Brand Comparison ---

سونی:
سونی به عنوان یک برند معتبر با نظرات مثبت زیادی در مورد کیفیت و بسته بندی محصولاتش شناخته می‌شود. با این حال، برخی مشتریان از